# Email assistant workflow with tool calling

## 1. Introduction

### 1.1 Lab overview

An assistant that can use multiple tools is able to carry out complex tasks, for example managing a calendar. In this lab, you'll explore a similar example - working with an **email assistant agent**.

This tool-calling workflow can carry out various tasks related to email management, including sending emails, searching for emails from a specific sender, and deleting emails. You'll give it natural language instructions - like "check unread emails from my boss" or "delete the Happy Hour email" - and see how it selects the right tools and completes the task for you.

An agent is an LLM plus tools, memory, actions, and guardrails. This lab uses an LLM plus tools, so it is a tool-calling workflow; the name "email assistant agent" is a short way to refer to it.

### 1.2 Learning outcome
By the end of this lab, you will be able to **connect an LLM to tools** with the Anthropic SDK, give natural language instructions, and observe how the model proposes tool calls, how your code runs them, and how the tool results go back to the model in multi-step tasks such as searching, sending, and deleting emails.

**Exercises.** Five short exercises are marked with `# TODO` comments: write a tool (Exercise 1), write the prompt builder (Exercise 2), choose the tools for a request without and with `delete_email` (Exercises 3 and 4), and read the tool trace of a run (Exercise 5). Complete each `# TODO` and run the check cell below it. The check cells run offline and do not call the model.

## 2. Initialize environment and client

You will now set up the environment. The next cell installs the Anthropic SDK, sets `MODEL` to `claude-sonnet-5-5`, reads the key from the Colab secret `AZURE_API_KEY`, and creates `client` for the Foundry endpoint. Add the secret with the key icon in the sidebar and turn on Notebook access before you run this cell; never paste the key into a cell. Expect only the usual pip messages, or a note about restarting the kernel.

In [ ]:
%pip install -q "anthropic>=1.4.0"
from anthropic import Anthropic
from google.colab import userdata

endpoint = "https://evergreen-ai-dev-resource.services.ai.azure.com/anthropic"
MODEL = "claude-sonnet-5-5"  # the deployment name
api_key = userdata.get("AZURE_API_KEY")  # Colab Secrets (key icon, left sidebar)

client = Anthropic(base_url=endpoint, api_key=api_key)

The next cell imports the libraries and the `beta_tool` decorator that the rest of the notebook uses. The cell prints nothing.

In [ ]:
# ================================
# Imports
# ================================
import json
from datetime import datetime
from html import escape
from typing import Optional

from anthropic import beta_tool
from IPython.display import HTML, display

### Define the tool-calling helpers

The next cell defines two helpers and prints nothing when it runs:

- `run_with_tools(prompt, tools, ...)` sends a request to Claude together with a list of Python functions. The SDK tool runner builds each tool definition (name, description, and `input_schema`) from the function's type hints and docstring. Claude only proposes the tool calls; the runner runs the tools for you, sends the tool results back, and repeats until Claude gives a final answer or the step limit (`max_iterations`) is reached. It returns the whole conversation as a list of messages.
- `show_trace(messages)` prints that conversation step by step: the request, each tool call with its arguments, each tool result, and the final answer.

Each tool in this lab is a function marked with `@beta_tool`, which does the same job as calling `beta_tool(f)` on a function, and the prompt is a plain string.

In [ ]:
def run_with_tools(prompt, tools, model=MODEL, max_iterations=10):
    """Let Claude call the Python tools on its own; return the whole message list."""
    messages = [{"role": "user", "content": prompt}]
    runner = client.beta.messages.tool_runner(
        model=model, max_tokens=2048, tools=tools,
        messages=messages, max_iterations=max_iterations)
    for reply in runner:                                      # one pass = one model call
        messages.append({"role": "assistant", "content": reply.content})
        tool_results = runner.generate_tool_call_response()   # runs the tools Claude asked for
        if tool_results:
            messages.append(tool_results)
    return messages


def show_trace(messages):
    """Print each step of a tool-calling run: text, tool calls, tool results, final answer."""
    def blocks(content):
        if isinstance(content, str):
            return [{"type": "text", "text": content}]
        return [b if isinstance(b, dict) else b.model_dump() for b in content]

    step = 0
    for m in messages:
        for b in blocks(m["content"]):
            if b["type"] == "text" and m["role"] == "user":
                print(f"USER: {b['text']}")
            elif b["type"] == "text":
                print(f"ASSISTANT: {b['text']}")
            elif b["type"] == "tool_use":
                step += 1
                print(f"[{step}] TOOL CALL  {b['name']}({json.dumps(b['input'])})")
            elif b["type"] == "tool_result":
                out = b["content"] if isinstance(b["content"], str) else "".join(c.get("text", "") for c in b["content"])
                print(f"    TOOL RESULT {out[:300]}")
            # "thinking" blocks are skipped on purpose

## 3. Simulated email service

### 3.1 Components
This lab uses a **simulated email service** to mimic real-world email interactions.
Think of it as your personal sandbox email inbox: it comes preloaded with messages so you can practice without sending real emails. Everything lives in memory inside this notebook, so nothing leaves your machine.

Instead of building this service from scratch, you'll be using tools that let you interact with it directly. Behind the scenes, it uses:

| Layer                          | Purpose                                              |
|--------------------------------|------------------------------------------------------|
| **Python class (`EmailService`)** | Implements each email action on an in-memory list |
| **Plain dictionaries**         | Hold each email: id, sender, recipient, subject, body, timestamp, read |
| **Type hints and docstrings**  | Describe valid inputs and become the tool definitions |
| **Anthropic tools**            | Bridge between the LLM and the service               |


### 3.2 Service functions
The service provides several functions that simulate common email actions.
Later on, these will be wrapped as tools so the assistant can use them automatically:

- `send(recipient, subject, body)` -> send a new email
- `list_emails()` -> list all emails
- `list_unread()` -> show only unread emails
- `get(email_id)` -> fetch a specific email by ID
- `search(query)` -> search emails by keyword
- `filter(recipient, start_date, end_date)` -> filter by recipient or date range
- `mark_read(email_id)` -> mark an email as read
- `mark_unread(email_id)` -> mark an email as unread
- `delete(email_id)` -> delete an email by ID
- `reset()` -> reset emails to initial state (for testing)


> **Key idea:** These service functions act as the "inbox controls."
> In the next steps, you'll expose them as Python functions (tools) that the LLM can call - turning plain functions into agent actions.

The next cell defines the preloaded emails and the `EmailService` class, and creates one service named `service`. It prints nothing. Running it again restores the inbox to its initial state.

In [ ]:
# ================================
# Simulated email service (in memory)
# ================================
INITIAL_EMAILS = [
    {"id": 1, "sender": "eric@work.com", "recipient": "you@email.com", "subject": "Happy Hour",
     "body": "We're planning drinks this Friday!", "timestamp": "2025-06-13T04:48:59.096908", "read": False},
    {"id": 2, "sender": "alice@work.com", "recipient": "you@email.com", "subject": "Lunch next week",
     "body": "Are you free for lunch on Tuesday?", "timestamp": "2025-06-13T09:30:12.481231", "read": False},
    {"id": 3, "sender": "boss@email.com", "recipient": "you@email.com", "subject": "Project status update",
     "body": "Please send me a short status update on the project by end of day.",
     "timestamp": "2025-06-13T11:05:41.220918", "read": False},
    {"id": 4, "sender": "boss@email.com", "recipient": "you@email.com", "subject": "Team meeting notes",
     "body": "Here are the notes from this morning's meeting.", "timestamp": "2025-06-13T13:42:07.003514", "read": True},
    {"id": 5, "sender": "newsletter@news.example.com", "recipient": "you@email.com", "subject": "Weekly digest",
     "body": "Here are this week's top stories.", "timestamp": "2025-06-13T15:10:30.775002", "read": True},
]


class EmailService:
    """A small in-memory email service. Each email is a dictionary."""

    MY_ADDRESS = "you@email.com"

    def __init__(self):
        self.reset()

    def reset(self):
        """Restore the inbox to its initial preloaded state."""
        self.emails = [dict(e) for e in INITIAL_EMAILS]
        self.next_id = max(e["id"] for e in self.emails) + 1
        return {"message": "Database reset", "count": len(self.emails)}

    def send(self, recipient, subject, body):
        email = {"id": self.next_id, "sender": self.MY_ADDRESS, "recipient": recipient, "subject": subject,
                 "body": body, "timestamp": datetime.now().isoformat(), "read": False}
        self.next_id += 1
        self.emails.append(email)
        return dict(email)

    def list_emails(self):
        newest_first = sorted(self.emails, key=lambda e: e["timestamp"], reverse=True)
        return [dict(e) for e in newest_first]

    def list_unread(self):
        return [e for e in self.list_emails() if not e["read"]]

    def _find(self, email_id):
        return next((e for e in self.emails if e["id"] == email_id), None)

    def get(self, email_id):
        email = self._find(email_id)
        return dict(email) if email else {"error": f"Email {email_id} not found"}

    def search(self, query):
        q = query.lower()
        return [e for e in self.list_emails()
                if q in e["subject"].lower() or q in e["body"].lower() or q in e["sender"].lower()]

    def filter(self, recipient=None, start_date=None, end_date=None):
        found = self.list_emails()
        if recipient:
            found = [e for e in found if e["recipient"].lower() == recipient.lower()]
        if start_date:
            found = [e for e in found if e["timestamp"][:10] >= start_date[:10]]
        if end_date:
            found = [e for e in found if e["timestamp"][:10] <= end_date[:10]]
        return found

    def _set_read(self, email_id, read):
        email = self._find(email_id)
        if email is None:
            return {"error": f"Email {email_id} not found"}
        email["read"] = read
        return dict(email)

    def mark_read(self, email_id):
        return self._set_read(email_id, True)

    def mark_unread(self, email_id):
        return self._set_read(email_id, False)

    def delete(self, email_id):
        email = self._find(email_id)
        if email is None:
            return {"error": f"Email {email_id} not found"}
        self.emails.remove(email)
        return {"deleted": True, "id": email_id}


service = EmailService()

### 3.3 Service test helpers
Before giving control to the LLM, you'll first **test the service yourself**.
The `test_*` functions are quick wrappers around the service functions. They let you try actions like **send, list, search, filter, mark, and delete** without calling the service class directly.

_**Note:** the helpers are defined in the next cell. It also defines `print_html`, which shows text or JSON in a titled box, and `reset_database()`, which restores the inbox. The cell prints nothing when it runs._

In [ ]:
def print_html(content, title=None):
    """Show text or JSON in a titled box."""
    if not isinstance(content, str):
        content = json.dumps(content, indent=2)
    else:
        try:
            content = json.dumps(json.loads(content), indent=2)
        except ValueError:
            pass
    heading = f"<div style='font-weight:600; margin-bottom:6px;'>{escape(title)}</div>" if title else ""
    display(HTML(
        "<div style='border:1px solid #bbb; border-radius:6px; padding:10px 12px; background:#f6f8fa; color:#1f2328;'>"
        f"{heading}<pre style='margin:0; white-space:pre-wrap; background:transparent; color:inherit;'>{escape(content)}</pre></div>"))


def _show(title, result):
    print_html(content=result, title=title)
    return result


def test_send_email():
    return _show("Send a test email", service.send("test@example.com", "Test email", "This is a test email."))

def test_get_email(email_id):
    return _show(f"Get email {email_id}", service.get(email_id))

def test_list_emails():
    return _show("List all emails", service.list_emails())

def test_filter_emails(recipient=None, start_date=None, end_date=None):
    return _show("Filter emails", service.filter(recipient, start_date, end_date))

def test_search_emails(query):
    return _show(f"Search emails for '{query}'", service.search(query))

def test_unread_emails():
    return _show("List unread emails", service.list_unread())

def test_mark_read(email_id):
    return _show(f"Mark email {email_id} as read", service.mark_read(email_id))

def test_mark_unread(email_id):
    return _show(f"Mark email {email_id} as unread", service.mark_unread(email_id))

def test_delete_email(email_id):
    return _show(f"Delete email {email_id}", service.delete(email_id))

def reset_database():
    result = service.reset()
    print(f"Email service reset to its initial state ({result['count']} emails).")

Each helper has a clear, self-explanatory name. In the next cell, you can uncomment the one you want to run, check its output, and confirm that the email service behaves as expected.

For example, you can test:
- Send a test email
- Fetch email by ID
- List all messages
- Mark email as read or unread
- Delete email

This step is your **sanity check** before handing the controls over to the agent. It gives you confidence that the simulated email service is alive and behaving just like a real email service would.

To test the service, **uncomment** the lines you'd like to try out, run the cell, and check the results instantly.

In [ ]:
# uncomment the line 'test_*' you want to try
new_email_id = test_send_email()
_ = test_get_email(new_email_id['id'])
#_ = test_list_emails()
#_ = test_filter_emails(recipient="test@example.com")
#_ = test_search_emails("lunch")
#_ = test_unread_emails()
#_ = test_mark_read(new_email_id['id'])
#_ = test_mark_unread(new_email_id['id'])
#_ = test_delete_email(new_email_id['id'])
#_ = reset_database()

## 4. Tool layer for the email agent

### 4.1 Why tools?
Now that the service is working, the next step is to expose it to the LLM as Python functions called **tools**. Each tool wraps a service function, transforming a raw call into an action the agent can perform - like list, read, search, send, delete, or toggle read.

> Think of tools as the agent's **actuators**: you give a natural language instruction ("check unread emails from my boss and send a polite reply"), and the model proposes **which tools** to call and **in what order** to complete the task.

### 4.2 Design hints
- Keep tool **docstrings** short, imperative, and specific to the action.
- Add **type hints** and an `Args:` section to the docstring: the Anthropic SDK turns them into the tool definition that Claude reads. The tool runner also checks the arguments that Claude writes against these type hints; if they do not fit, it sends an error back as the tool result instead of running the tool.
- Return **consistent, compact JSON text** (the tool result as data) so the model can chain results and write the wording for the user. Return errors as data too: the service answers `{"error": "Email 99 not found"}` for an unknown id, and the model reads it and can fix its call.
- Prefer **one clear responsibility per tool** (single service call, single effect).

### 4.3 Available tools
| Tool Function                      | Action                                                                 |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()`                | Fetch all emails, newest first                                         |
| `list_unread_emails()`             | Retrieve only unread emails                                            |
| `search_emails(query)`             | Search by keyword in subject, body, or sender                          |
| `filter_emails(...)`               | Filter by recipient and/or date range                                  |
| `get_email(email_id)`              | Fetch a specific email by ID                                           |
| `mark_email_as_read(id)`           | Mark an email as read                                                  |
| `mark_email_as_unread(id)`         | Mark an email as unread                                                |
| `send_email(...)`                  | Send a new (simulated) email                                           |
| `delete_email(id)`                 | Delete an email by ID                                                  |
| `search_unread_from_sender(addr)`  | Return unread emails from a given sender (e.g., `boss@email.com`)      |

**Note:** the next cell defines nine of the ten tools. Each one is a plain Python function marked with `@beta_tool`, and it returns its result as compact JSON text. The cell prints nothing when it runs. You write the tenth tool, `search_unread_from_sender`, in Exercise 1.

In [ ]:
def _as_json(data) -> str:
    """Return data as compact JSON text (a tool result is text)."""
    return json.dumps(data, separators=(",", ":"))


@beta_tool
def list_all_emails() -> str:
    """List all emails, newest first."""
    return _as_json(service.list_emails())


@beta_tool
def list_unread_emails() -> str:
    """List only the unread emails."""
    return _as_json(service.list_unread())


@beta_tool
def search_emails(query: str) -> str:
    """Search emails by keyword in the subject, body, or sender.

    Args:
        query: The keyword or phrase to search for.
    """
    return _as_json(service.search(query))


@beta_tool
def filter_emails(recipient: Optional[str] = None, start_date: Optional[str] = None,
                  end_date: Optional[str] = None) -> str:
    """Filter emails by recipient and/or date range.

    Args:
        recipient: Only return emails sent to this address.
        start_date: Only return emails on or after this date (YYYY-MM-DD).
        end_date: Only return emails on or before this date (YYYY-MM-DD).
    """
    return _as_json(service.filter(recipient, start_date, end_date))


@beta_tool
def get_email(email_id: int) -> str:
    """Get a specific email by its ID.

    Args:
        email_id: The ID of the email.
    """
    return _as_json(service.get(email_id))


@beta_tool
def mark_email_as_read(email_id: int) -> str:
    """Mark an email as read.

    Args:
        email_id: The ID of the email.
    """
    return _as_json(service.mark_read(email_id))


@beta_tool
def mark_email_as_unread(email_id: int) -> str:
    """Mark an email as unread.

    Args:
        email_id: The ID of the email.
    """
    return _as_json(service.mark_unread(email_id))


@beta_tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send a new email.

    Args:
        recipient: The email address to send to.
        subject: The subject line.
        body: The text of the email.
    """
    return _as_json(service.send(recipient, subject, body))


@beta_tool
def delete_email(email_id: int) -> str:
    """Delete an email by its ID.

    Args:
        email_id: The ID of the email.
    """
    return _as_json(service.delete(email_id))

### Write a tool

**Exercise 1.** Complete the tool `search_unread_from_sender`, which returns the unread emails from one sender, for example `boss@email.com`, whatever the case of the letters. The parameter is named `sender`. Write the type hints, a docstring with a one-line description and an `Args:` section, and the body. Like the other tools, it returns compact JSON text. Requires the previous cell. Expect no output.

In [ ]:
# TODO 1: Add the type hints and a docstring with an Args section, then write the body.
# Hint 1: The SDK builds the tool schema from the type hints and the docstring, and Claude reads that schema.
# Hint 2: service.list_unread() returns the unread emails; _as_json turns a result into text.
@beta_tool
def search_unread_from_sender(sender):
    # YOUR CODE HERE (type hints, a docstring with an Args section, and the body)
    return "[]"

### Check your work

This cell defines `check`, `tool_name`, and `fresh_inbox`, which test your exercises without a model call. `fresh_inbox` runs a test against a new inbox with the preloaded emails, so your own inbox is never changed. Expect no output.

In [ ]:
# Helpers for the check cells; no model call is made.
def check(n, test, expected):
    """Runs one offline test and prints a message when it fails."""
    try:
        passed, why = bool(test()), ""
    except Exception as e:
        passed, why = False, f" ({type(e).__name__})"
    if not passed:
        print(f"Exercise {n} is incomplete. Expected: {expected}{why}")
    return passed

def tool_name(t):
    """Returns the name of a tool, or of a plain function."""
    return getattr(t, "name", None) or getattr(t, "__name__", str(t))

def fresh_inbox(test):
    """Runs test() against a new inbox, then puts your own inbox back."""
    global service
    kept = service
    service = EmailService()
    try:
        return test()
    finally:
        service = kept

This cell checks Exercise 1 offline: the tool name, the description, the schema of `sender`, and the results on the preloaded inbox. Expect a pass message or the open exercise.

In [ ]:
# Check for Exercise 1; no model call is made.
spec = lambda: search_unread_from_sender.to_dict()
ids = lambda s: [e["id"] for e in json.loads(search_unread_from_sender(s))]
prop = lambda: spec()["input_schema"]["properties"]["sender"]

done = [
    check(1, lambda: tool_name(search_unread_from_sender) == "search_unread_from_sender" and callable(spec),
          "search_unread_from_sender is marked with @beta_tool."),
    check(1, lambda: len(spec()["description"].strip()) >= 10,
          "the function has a docstring that says what the tool does."),
    check(1, lambda: prop()["type"] == "string" and spec()["input_schema"]["required"] == ["sender"],
          "the parameter sender has the type hint str."),
    check(1, lambda: prop().get("description"),
          "the docstring has an Args section that describes sender."),
    check(1, lambda: fresh_inbox(lambda: isinstance(json.loads(search_unread_from_sender("boss@email.com")), list)),
          "the tool returns JSON text that holds a list of emails."),
    check(1, lambda: fresh_inbox(lambda: ids("boss@email.com") == [3] and ids("eric@work.com") == [1]
                                 and ids("newsletter@news.example.com") == [] and ids("nobody@email.com") == []
                                 and ids("email.com") == [] and ids("boss") == []),
          "only the unread emails from that sender (email 4 from boss@email.com is already read), and none for a part of an address such as email.com."),
    check(1, lambda: fresh_inbox(lambda: ids("BOSS@Email.com") == [3]),
          "the sender matches whatever the case of the letters."),
    check(1, lambda: fresh_inbox(lambda: (search_unread_from_sender("boss@email.com"), len(service.list_unread()))[1] == 3),
          "the tool only reads the inbox and changes no email."),
]
print("Exercise 1: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")

A tool marked with `@beta_tool` can still be called like a normal Python function, so you can test it directly. For example, **testing `list_all_emails()` tool:**
```python
    all_emails = list_all_emails()
```

Now, let's try out a few tools that connect to the simulated service to make sure everything is working.

**Uncomment** the ones you'd like to run, execute the cell, and review the outputs.

In [ ]:
# Test sending a new email and fetch it by ID
new_email = json.loads(send_email("test@example.com", "Lunch plans", "Shall we meet at noon?"))
content_ = get_email(new_email['id'])

# Uncomment the ones you want to try:
#content_ = list_all_emails()
#content_ = list_unread_emails()
#content_ = search_emails("lunch")
#content_ = filter_emails(recipient="test@example.com")
#content_ = mark_email_as_read(new_email['id'])
#content_ = mark_email_as_unread(new_email['id'])
#content_ = search_unread_from_sender("test@example.com")
#content_ = delete_email(new_email['id'])

print_html(content=content_, title="Testing the email tools")

## 5. Preparing the agent prompt

Before assigning tasks to the email assistant agent, you'll create a small helper function called `build_prompt()`.
This function wraps the natural language request in a system-style preamble so the LLM:

- Recognizes that it's acting as an **email assistant agent**
- Understands it has permission to use the available tools
- Executes actions directly, without asking for confirmation (no human-in-the-loop)

This is acceptable here only because the inbox is simulated. For real changes, such as deleting an email, the app should ask a person for approval before the tool runs.

You will write `build_prompt()` in Exercise 2.

### Write the prompt builder

**Exercise 2.** Complete `build_prompt()` so that it returns a preamble followed by the request. The preamble must say that the model is an email assistant, that it may use the provided tools, that it must never ask for confirmation, and that your address is `you@email.com`. The request comes last, without leading or trailing whitespace. Expect no output.

In [ ]:
# TODO 2: Return the preamble followed by the request.
# Hint 1: A triple-quoted f-string can hold several lines and insert request_.
# Hint 2: Each line of the preamble states one rule; strip() removes the whitespace around the request.
def build_prompt(request_: str) -> str:
    # YOUR CODE HERE
    return request_

This cell checks Exercise 2 offline by calling `build_prompt` with sample requests. Expect a pass message or the open exercise.

In [ ]:
# Check for Exercise 2; no model call is made.
import re

REQ = "Delete the Happy Hour email"
OTHER = "List my unread mail"

def never_asks(text):
    """True when every sentence that mentions confirmation also forbids asking for it."""
    parts = [s for s in re.split(r"[.!?;\n]", text.lower()) if "confirm" in s]
    return bool(parts) and all(re.search(r"\b(never|not|without|no)\b|n't", s) for s in parts)

def stripped():
    """True when whitespace around the request changes nothing in the prompt."""
    return all(build_prompt("  \n  " + r + "  \n\n") == build_prompt(r) and r in build_prompt(r) for r in (REQ, OTHER))

done = [
    check(2, lambda: all(build_prompt(r).strip().endswith(r) and build_prompt(r).strip()[:-len(r)].strip() != ""
                         for r in (REQ, OTHER)) and REQ not in build_prompt(OTHER),
          "a preamble followed by the request, with the request last, for any request."),
    check(2, lambda: "assistant" in build_prompt(REQ).lower() and "email" in build_prompt(REQ).lower(),
          "the preamble says that the model is an email assistant."),
    check(2, lambda: "tool" in build_prompt(REQ).lower(),
          "the preamble says that the model may use the provided tools."),
    check(2, lambda: never_asks(build_prompt(REQ)),
          "the preamble says that the model must never ask for confirmation."),
    check(2, lambda: "you@email.com" in build_prompt(REQ),
          "the preamble gives the address you@email.com."),
    check(2, stripped,
          "the request has no leading or trailing whitespace."),
]
print("Exercise 2: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")

Run the next cell to see how `build_prompt` **wraps your raw user prompt** with a preamble of instructions. Requires Exercise 2. Expect the preamble you wrote followed by the request.
For example:

In [ ]:
example_prompt = build_prompt("Delete the Happy Hour email")
print_html(content=example_prompt, title="Example example_prompt")

### 5.1 Resetting the email service

Since you've been experimenting with the email service, let's reset it back to its initial state.

You can do this by calling the helper function that clears and refreshes the simulated email service. Every scenario cell in section 6 also starts with `reset_database()`, so you can run any of them again and always start from the same inbox. The cell prints a short confirmation:

In [ ]:
reset_database()

## 6. LLM + Email tools

### 6.1 Scenario
So far, you've been working directly with the service. Now it's time to let the **LLM take control** as your email assistant agent.

For example, you might ask it:
> "Check for unread emails from `boss@email.com`, mark them as read, and send a polite follow-up."

### 6.2 What happens
1. The agent interprets your instruction.
2. It proposes the right tool calls (`search_unread_from_sender` -> `mark_email_as_read` -> `send_email`), one step at a time.
3. Your code, through the tool runner, runs each tool call automatically, without asking for confirmation, and sends each tool result back until the LLM gives a final answer or reaches the step limit (`max_iterations`).

The Anthropic SDK tool runner handles the tool definitions, the arguments, running the tools, and passing the tool results between steps - so you can focus on **what** the agent achieves, not **how** to call the service.

### Choose the tools

**Exercise 3.** Define `tools_without_delete`, the list of tools for the first request and for the request in 6.4, where `delete_email` is missing on purpose. Put the tool functions themselves in the list: `search_unread_from_sender`, `list_unread_emails`, `search_emails`, `get_email`, `mark_email_as_read`, and `send_email`. Do not include `delete_email`. Requires Exercise 1. Expect no output.

In [ ]:
# TODO 3: List the tools the assistant may use for the first request, without delete_email.
# Hint 1: The list holds the tool functions themselves, not their names in quotes.
# Hint 2: The tool you leave out is the one that deletes an email.
tools_without_delete = [
    # YOUR CODE HERE
]

This cell checks Exercise 3 offline by looking at the names of the tools in the list. Expect a pass message or the open exercise.

In [ ]:
# Check for Exercise 3; no model call is made.
names = lambda lst: [tool_name(t) for t in lst]
needed = {"search_unread_from_sender", "search_emails", "mark_email_as_read", "send_email"}

done = [
    check(3, lambda: isinstance(tools_without_delete, list) and len(tools_without_delete) > 0
          and not any(isinstance(t, str) for t in tools_without_delete),
          "tools_without_delete is a list of tool functions, not of names in quotes."),
    check(3, lambda: needed <= set(names(tools_without_delete)),
          "the list includes search_unread_from_sender, search_emails, mark_email_as_read, and send_email."),
    check(3, lambda: "delete_email" not in names(tools_without_delete),
          "the list does not include delete_email."),
    check(3, lambda: len(set(names(tools_without_delete))) == len(tools_without_delete),
          "every tool appears once."),
]
print("Exercise 3: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")

### 6.3 Run it
Run the next cell and see how the agent orchestrates multiple tools to fulfill your request. Requires Exercises 1 to 3. Each scenario cell starts by resetting the inbox, so you can run it again. Feel free to try your own requests with the email service as well.

*What to look for:*
- A clear **tool-call trace** showing which tools were used and the arguments passed.
- A concise **final message** summarizing the actions performed (e.g., "Found 1 unread email, marked as read, sent follow-up").

In [ ]:
reset_database()
if not tools_without_delete:
    print("Complete Exercises 1 to 3, then run this cell again.")
else:
    # Try your own requests
    prompt_ = build_prompt("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")

    messages = run_with_tools(
        prompt_,
        model=MODEL, # LLM
        tools=tools_without_delete, # list of tools that the LLM can access
        max_iterations=5,
    )

    show_trace(messages)


### 6.4 Missing tool: `delete_email`

What happens if the tool you need isn't available?

For example, try this request:
> "Delete alice@work.com email."

Since the `delete_email` tool is not registered, its tool definition is never sent to the LLM and the app runs only registered tools. The LLM will still attempt to respond, but it cannot act, so it explains why.

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">
  <h4>Key Insight</h4>
  <p style="margin: 0;">
    This highlights an important point: <b>the tools you make available directly determine what the agent can do.</b>
  </p>
</div>


**All tools defined in this lab** (the run below registers only six of them: `search_unread_from_sender`, `list_unread_emails`, `search_emails`, `get_email`, `mark_email_as_read`, and `send_email`)

| Tool Function                      | Action                                                                 |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()`                | Fetch all emails, newest first                                         |
| `list_unread_emails()`             | Retrieve only unread emails                                            |
| `search_emails(query)`             | Search by keyword in subject, body, or sender                          |
| `filter_emails(...)`               | Filter by recipient and/or date range                                  |
| `get_email(email_id)`              | Fetch a specific email by ID                                           |
| `mark_email_as_read(id)`           | Mark an email as read                                                  |
| `mark_email_as_unread(id)`         | Mark an email as unread                                                |
| `send_email(...)`                  | Send a new (simulated) email                                                |
| `delete_email(id)`                 | Delete an email by ID                                                  |
| `search_unread_from_sender(addr)`  | Return unread emails from a given sender (e.g., `boss@email.com`)      |

Run the next cell. It uses `tools_without_delete` from Exercise 3, so expect the assistant to explain that it cannot delete the email. The last line of the cell shows whether the email from alice@work.com is still in the inbox: expect `True`, because nothing was deleted.

In [ ]:
reset_database()
if not tools_without_delete:
    print("Complete Exercises 1 to 3, then run this cell again.")
else:
    # Try with a request that may call an unavailable tool
    prompt_ = build_prompt("Delete alice@work.com email")

    messages = run_with_tools(
        prompt_,
        model=MODEL,
        tools=tools_without_delete, # list of tools that the LLM can access
        max_iterations=5
    )

    show_trace(messages)
    print("Email from alice@work.com still in the inbox:", any(e["sender"] == "alice@work.com" for e in service.list_emails()))


#### 6.4.1 Try again with `delete_email` enabled

Now give the LLM the tool that was missing.

In the previous step of the lab, you gave the LLM access to the following list of tools (your `tools_without_delete`):
```python
    tools=[
        search_unread_from_sender,
        list_unread_emails,
        search_emails,
        get_email,
        mark_email_as_read,
        send_email
    ]
```

And because of the tools available in that list, the agent couldn't delete emails because the `delete_email` tool wasn't available.

Now add `delete_email` to the tool list. Exercise 4 below builds the longer list, `tools_with_delete`. Once its check passes, run the trace cell that follows it and compare the trace with the previous one. This time, the agent has everything it needs to finish the task.

> Tip: Watch the sequence of calls - after finding the target message, the agent should select `delete_email` to complete the action.

### Add the missing tool

**Exercise 4.** Define `tools_with_delete`: the same tools as `tools_without_delete`, plus `delete_email`. Requires Exercises 1 and 3. Expect no output.

In [ ]:
# TODO 4: List the same tools and add delete_email.
# Hint 1: The new list holds everything in tools_without_delete plus delete_email.
# Hint 2: A new list can be built from an existing one without retyping it.
tools_with_delete = [
    # YOUR CODE HERE
]

This cell checks Exercise 4 offline by comparing the names in the two lists. Expect a pass message or the open exercise.

In [ ]:
# Check for Exercise 4; no model call is made.
done = [
    check(4, lambda: isinstance(tools_with_delete, list) and len(tools_with_delete) > 0
          and not any(isinstance(t, str) for t in tools_with_delete),
          "tools_with_delete is a list of tool functions, not of names in quotes."),
    check(4, lambda: "delete_email" in names(tools_with_delete),
          "the list includes delete_email."),
    check(4, lambda: set(names(tools_without_delete)) <= set(names(tools_with_delete))
          and "delete_email" not in names(tools_without_delete),
          "the list has every tool of tools_without_delete (Exercise 3 must not hold delete_email)."),
    check(4, lambda: len(set(names(tools_with_delete))) == len(tools_with_delete),
          "every tool appears once."),
]
print("Exercise 4: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")

This cell runs the same request again with `tools_with_delete`. Requires Exercises 1 to 4. Expect a trace in which the assistant finds the email from alice@work.com and then calls `delete_email`. The last line of the cell shows whether the email from alice@work.com is still in the inbox: expect `False`.

In [ ]:
reset_database()
if not tools_with_delete:
    print("Complete Exercises 1 to 4, then run this cell again.")
else:
    prompt_ = build_prompt("Delete alice@work.com email")

    messages = run_with_tools(
        prompt_,
        model=MODEL,
        tools=tools_with_delete,
        max_iterations=5
    )

    show_trace(messages)
    print("Email from alice@work.com still in the inbox:", any(e["sender"] == "alice@work.com" for e in service.list_emails()))


### 6.5 Targeted action: Delete the "Happy Hour" email

The test inbox comes preloaded with a message titled **"Happy Hour."**  Your task is to instruct the agent to locate this message and delete it.

For reference, here's the entry from the simulated inbox:

```json
{
  "id": 1,
  "sender": "eric@work.com",
  "recipient": "you@email.com",
  "subject": "Happy Hour",
  "body": "We're planning drinks this Friday!",
  "timestamp": "2025-06-13T04:48:59.096908",
  "read": false
}
```
Run the next cell and watch how the agent searches for the message and deletes it from the inbox. Requires Exercises 1 to 4.

In [ ]:
reset_database()
if not tools_with_delete:
    print("Complete Exercises 1 to 4, then run this cell again.")
else:
    prompt_ = build_prompt("Delete the happy hour email")

    messages = run_with_tools(
        prompt_,
        model=MODEL,
        tools=tools_with_delete,
        max_iterations=5
    )

    show_trace(messages)


### Confirm the tool trace

**Exercise 5.** Write `tools_used` to return the names of the tools that a run called, in order, with one entry per call. `messages` is the list that `run_with_tools` returns. A tool call is a content block of type `tool_use`. A message's content can be plain text or a list of blocks, and a block can be a dictionary or an SDK object. Expect no output.

In [ ]:
# TODO 5: Return the name of every tool call in the messages, in order.
# Hint 1: A content block that asks for a tool has the type "tool_use" and a name.
# Hint 2: A message's content is plain text or a list of blocks; a block is a dictionary or an object, and model_dump() turns an object into a dictionary.
def tools_used(messages):
    """Returns the names of the tools that were called, in order, one entry per call."""
    # YOUR CODE HERE
    return []

This cell applies `tools_used` to the Happy Hour run above and then looks at the inbox. Requires Exercise 5 and the previous run. Expect `delete_email` in the list of tools used, usually after `search_emails`, and `False` on the last line.

In [ ]:
if "messages" not in globals() or "happy hour" not in str(messages[0]["content"]).lower():
    print("Run the Happy Hour cell above first, then run this cell again.")
else:
    used = tools_used(messages)
    print("Tools used:", used)
    print("Happy Hour still in the inbox:", any(e["subject"] == "Happy Hour" for e in service.list_emails()))

This cell checks Exercise 5 offline with made-up messages that hold dictionary blocks, SDK objects, plain text, and repeated calls. Expect a pass message or the open exercise.

In [ ]:
# Check for Exercise 5; no model call is made.
from anthropic.types.beta import BetaTextBlock, BetaToolUseBlock

def call(name, i):
    return {"type": "tool_use", "id": f"t{i}", "name": name, "input": {}}

def result(i):
    return {"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"t{i}", "content": "[]"}]}

mixed = [
    {"role": "user", "content": "Delete the Happy Hour email"},
    {"role": "assistant", "content": [
        {"type": "thinking", "thinking": "Search first.", "signature": "x"},
        BetaTextBlock(type="text", text="I will search first.", citations=None),
        BetaToolUseBlock(type="tool_use", id="t1", name="search_emails", input={"query": "happy hour"})]},
    result(1),
    {"role": "assistant", "content": [call("delete_email", 2)]},
    result(2),
    {"role": "assistant", "content": [{"type": "text", "text": "Done."}]},
]
repeated = [
    {"role": "user", "content": "x"},
    {"role": "assistant", "content": [call("search_emails", 1)]}, result(1),
    {"role": "assistant", "content": [call("search_emails", 2), call("send_email", 3)]},
    result(2),
    {"role": "assistant", "content": [{"type": "text", "text": "Done."}]},
]
no_tools = [{"role": "user", "content": "Hello"},
            {"role": "assistant", "content": [{"type": "text", "text": "Hi."}]}]

done = [
    check(5, lambda: tools_used(mixed) == ["search_emails", "delete_email"],
          "the names of the tool_use blocks in order, from dictionary blocks and from SDK objects, skipping plain-text messages and tool results."),
    check(5, lambda: tools_used(repeated) == ["search_emails", "search_emails", "send_email"],
          "one entry per call: a tool called twice appears twice, in the order of the calls."),
    check(5, lambda: tools_used(no_tools) == [],
          "an empty list when no tool was called."),
]
print("Exercise 5: all checks passed." if all(done)
      else "Complete the open exercise, then run this cell again.")

## 7. Final Takeaways

- In this lab, **you explored** how an LLM email agent interacts with a simulated email service using tool calls.
- **Tool calling** allows LLMs to go beyond text generation - they propose calls to functions (tools) that your code runs, and the tool results go back to the model, so it can complete multi-step tasks.
- The set of available tools determines what the agent can and cannot do (e.g., without `delete_email`, it cannot remove messages). Only registered tools run, and a change such as deleting an email would normally need approval from a person first.
- Clear docstrings and consistent behavior help the LLM select the right tool for each step.
- The Anthropic SDK tool runner manages the interaction layer: turning Python functions into tool definitions, passing the model's arguments to your functions, and returning the tool results to the model, until the final answer or the step limit.
- Tools should return compact JSON text as data, including clear errors, so that the model can chain results and fix its own calls.
- Observing the full workflow - from prompt -> tool calls -> outputs -> final response - is key to understanding and improving how agents reason and act.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

<strong>Congratulations!</strong>

You just guided an LLM-powered <em>email assistant agent</em> from natural language requests to concrete actions on a simulated email service. You saw how tools turn plain language into reliable operations - listing, searching, marking as read, sending, and deleting - without you touching the service code directly.

You learned that the tools you expose define the agent's real capabilities. When a tool is missing, the agent can reason but cannot act; when it's present, it can chain steps, pass parameters, and deliver exactly what you asked for. Along the way, the Anthropic SDK tool runner handled the plumbing: the tool definitions, the arguments, running the tools, and passing the tool results - so you could focus on what you wanted done, not how to wire it up.

With this workflow under your belt, you're ready to design task-focused assistants with a small set of clear tools, tool results returned as data, and a step limit, and to add argument checks and approval before changes such as deleting emails.
</div>